# 11 — Knowledge Agent (`agents/knowledge_agent.py`)
RAG (Retrieval-Augmented Generation) agent that retrieves business context and definitions.

Flow:
1. `similarity_search(store, query, k=5)` — vector similarity search
2. Filter: `score >= 0.70`
3. Return structured `AgentResult` with `data={"knowledge": [entries]}`

In mock mode (default), uses `_NullVectorStore` — no DB or API key needed.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)
os.environ["ENABLE_MOCK"] = "true" 

## 1. Initialize KnowledgeAgent

In [ ]:
from agents.knowledge_agent import KnowledgeAgent
from config.settings import config

agent = KnowledgeAgent(config=config)
print("Agent name :", agent.name)
print("Description:", agent.description)
print("Store type :", type(agent._store).__name__)
print("Capabilities:", agent.capabilities)

## 2. Basic Query — Retention Definitions

In [ ]:
from core.base_agent import AgentRequest

req = AgentRequest(
    query="What is gross retention rate and how is it calculated?",
    intent="knowledge_lookup",
    data_products=["retention"],
)
result = agent.execute(req)

print("success   :", result.success)
print("confidence:", result.confidence)
print("sources   :", result.sources)
print("\n--- Summary ---")
print(result.summary[:500])

## 3. Knowledge Data Payload

In [ ]:
# result.data = {"knowledge": [list of entry dicts]}
if result.data and "knowledge" in result.data:
    entries = result.data["knowledge"]
    print(f"Returned {len(entries)} knowledge entries:\n")
    for e in entries:
        print(f"  Topic  : {e['topic']}")
        print(f"  Source : {e['source']}")
        print(f"  Def    : {e['definition'][:80]}...")
        print()

## 4. Query for Different Products

In [ ]:
queries = [
    ("CAC payback period formula", "cac"),
    ("LTV:CAC ratio benchmark", "ltv"),
    ("What are bookings and ARR?", "bookings"),
    ("Data quality rules enforcement", None),
]

for query, product in queries:
    req = AgentRequest(
        query=query,
        data_products=[product] if product else [],
    )
    r = agent.execute(req)
    entry_count = len(r.data.get("knowledge", [])) if r.data else 0
    print(f"Query: '{query[:50]}'")
    print(f"  confidence={r.confidence:.2f} | entries={entry_count} | success={r.success}")

## 5. No Results Case

In [ ]:
req = AgentRequest(query="xyz abc 999 completely irrelevant gibberish")
# NullVectorStore always returns some results, but let's check the threshold logic
r = agent.execute(req)
print("success   :", r.success)
print("confidence:", r.confidence)
entries = r.data.get("knowledge", []) if r.data else []
print("entries   :", len(entries))
print("summary   :", r.summary[:200])

## 6. AgentResult.to_dict() for API serialization

In [ ]:
import json
req = AgentRequest(query="What is NRR?")
r = agent.execute(req)
d = r.to_dict()
# Show just top-level keys
print("Result keys:", list(d.keys()))
print("confidence :", d["confidence"])
print("sources    :", d["sources"][:2])
print("data keys  :", list(d["data"].keys()) if d["data"] else None)